# Paper 3, pass 2: does the checkpoint predict these recordings?

Pass 1 measured the ceiling from the recordings alone. Its first result was
withdrawn: it read `list(f.keys())[0]`, which is episode `s01e02a` rather than
the episode this pass predicts, and its transpose guard could not fire on a
`(482, 1000)` array, so every correlation ran across the parcel axis. The
corrected ceiling for `s01e01a` is `0.1517`, CI `[0.1484, 0.1551]`, four
subjects, 1000 of 1000 parcels, 592 timepoints, from
`scripts/paper3_noise_ceiling.py`. The withdrawn figure was `0.2247`.

This pass runs the released checkpoint over the same stimulus and correlates
its predictions against those recordings, matching the episode by name and
asserting orientation against the atlas size.

Committed before the run: the sign is the finding, nothing is flipped or
absolute-valued, and a parcel-level `r` is not comparable with the audit's
vertex-level `-0.0145` because averaging within a parcel raises correlations.

**Attach `ckadirt/algonauts2025nsl`. GPU required.**

In [ ]:
%%bash
# Same pinned build as the corpus scan: tribev2 declares torch>=2.5.1,<2.7 and Kaggle
# ships newer, so the model would otherwise run against a version it was never built for.
set -e
nvidia-smi --query-gpu=name --format=csv,noheader | head -1 | sed 's/^/card: /'
pip install -q --index-url https://download.pytorch.org/whl/cu121 \
  torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
python -c "import torch; print('torch', torch.__version__)"

In [ ]:
import glob
import os

roots = sorted(glob.glob('/kaggle/input/**/algonauts_2025.competitors', recursive=True))
root = roots[0]
print('root:', root)

# The stimulus this pass predicts. Find it before spending GPU time on setup: if the
# media is not present in the mirror, the run should stop here rather than at hour two.
media = sorted(
    glob.glob(os.path.join(root, 'stimuli', '**', '*.mkv'), recursive=True)
    + glob.glob(os.path.join(root, 'stimuli', '**', '*.mp4'), recursive=True)
)
print(f'stimulus files found: {len(media)}')
for path in media[:5]:
    print(f'  {os.path.getsize(path)/1e6:8.1f} MB  {os.path.relpath(path, root)}')

assert media, 'No stimulus media in this mirror. Pass 2 cannot run; report that and stop.'

# The heavy steps run in a subprocess, which cannot see these variables, so the two paths
# it needs are handed over through the environment.
# The published ceiling is for s01e01a. The script recomputes a ceiling from whatever
# episode it is given, so any stimulus yields a self-consistent answer, but picking that
# episode is what makes the number directly comparable with the 0.1517 already reported.
CEILING_EPISODE = 's01e01a'
preferred = [p for p in media if CEILING_EPISODE in os.path.basename(p)]
stimulus = preferred[0] if preferred else media[0]
if not preferred:
    print(f'note: {CEILING_EPISODE} not in this mirror, falling back to {os.path.basename(stimulus)}')

os.environ['MONARCH_STIMULUS'] = stimulus
os.environ['MONARCH_H5_DIR'] = os.path.join(root, 'fmri')

# Set to a number of seconds to predict only that much of the episode, or leave empty for
# the whole thing. The first run of a new configuration should be short: it measures the
# per-chunk cost, which is the number that decides whether the full episode fits the
# 12-hour session cap at all.
MAX_SECONDS = '120'
os.environ['MONARCH_MAX_SECONDS'] = MAX_SECONDS

print('stimulus:', os.environ['MONARCH_STIMULUS'])
print('h5 dir  :', os.environ['MONARCH_H5_DIR'])
print('max seconds:', MAX_SECONDS or '(whole episode)')


In [ ]:
%%bash
set -e
mkdir -p /kaggle/temp
cd /kaggle/temp
rm -rf monarch tribev2
git clone -q --branch thesis/amendment-and-analysis-layer \
  https://github.com/brn-mwai/monarch.git monarch
git clone -q https://github.com/brn-mwai/tribev2.git
apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y ffmpeg > /dev/null
printf 'torch==2.5.1\ntorchvision==0.20.1\ntorchaudio==2.5.1\n' > /kaggle/temp/constraints.txt
pip install -q exca==0.5.21
pip install -q -c /kaggle/temp/constraints.txt /kaggle/temp/tribev2
pip install -q -c /kaggle/temp/constraints.txt "whisperx==3.4.2"
pip install -q -c /kaggle/temp/constraints.txt nltk nibabel ujson mne torchmetrics
pip install -q --no-deps "ctranslate2==4.5.0"
python -c "import neuralset, neuraltrain, tribev2; print('tribev2 stack imports OK')"

In [ ]:
import sys

sys.path.insert(0, '/kaggle/temp/monarch/services/inference')
from scripts.kaggle_bootstrap import apply_session_environment

SESSION = apply_session_environment()

In [ ]:
%%bash
# Pass 2 runs as a subprocess, not in the kernel, and that is the whole point.
#
# v1 and v2 both died at the first in-kernel import of the stack with
# "cannot import name '_center' from 'numpy._core.umath'". Installing the pinned
# requirements upgrades numpy on disk, while this kernel process still holds the image's
# numpy in memory, so scipy reads a new strings.py against an old compiled umath. A
# subprocess starts after the installs and sees one numpy, which is why every heavy step in
# the corpus scan is shelled out the same way.
#
# --work-dir because tribev2 writes the extracted .wav beside the video, and every Kaggle
# dataset is mounted read-only. v3 reached the model and died there.
#
# python -u is what v6 was missing. It ran 12 hours, was killed at the session cap, and
# published a log that stopped at "session environment ready". tqdm writes the progress bar
# to stderr, the subprocess is not a tty, so Python block-buffered every line and the kill
# took the buffer with it. Twelve GPU-hours, no evidence. The flag costs nothing.
#
# MONARCH_MAX_SECONDS, when set, trims the stimulus. The recordings are truncated to match,
# so a short window is a valid comparison on fewer timepoints rather than a broken one.
set -e
cd /kaggle/temp/monarch/services/inference
PYTHONPATH=/kaggle/temp/tribev2 python -u scripts/paper3_prediction.py \
  --stimulus "$MONARCH_STIMULUS" \
  --h5-dir "$MONARCH_H5_DIR" \
  --out /kaggle/working/paper3_validation.json \
  --prediction-out /kaggle/working/prediction.npy \
  --work-dir /kaggle/temp/stimulus \
  ${MONARCH_MAX_SECONDS:+--max-seconds $MONARCH_MAX_SECONDS}


In [ ]:
import json

with open('/kaggle/working/paper3_validation.json') as handle:
    result = json.load(handle)

print(json.dumps(result, indent=2))


## Reading this

Whatever the sign, it is reported as measured. A negative encoder correlation
against a positive ceiling would replicate the audit's direction in parcel
space; a positive one below the ceiling means the checkpoint carries signal but
less than people share with each other.
